In [ ]:
RUN_NAME = "test"
EPOCHS = 1
BATCH_SIZE = 32
CONTEXT_LENGTH = 128
VOCAB_SIZE = 50257
NUM_LAYERS = 6
D_MODEL = 512
D_FEEDFORWARD = 2048
NUM_HEADS = 8
LR = 0.0001

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_pretrained("gpt2")
tokenizer.get_vocab_size()

In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import Dataset
from tokenizers import Tokenizer

class WikiText103(Dataset):
    def __init__(self, split: str, tokenizer: Tokenizer, context_length: int):
        super().__init__()
        self.split = split
        self.tokenizer = tokenizer
        self.context_length = context_length
        dataset = load_dataset("iohadrubin/wikitext-103-raw-v1")
        tokens = []
        for text in dataset[split]["text"]:
            tokens += list(self.tokenizer.encode(text).ids)
        self.data = torch.tensor(tokens, dtype=torch.uint16)
    def __len__(self):
        return len(self.data)//self.context_length
    def __getitem__(self, idx):
        return self.data[idx*self.context_length:idx*self.context_length+self.context_length].long(), self.data[idx*self.context_length+1:idx*self.context_length+self.context_length+1].long()

In [ ]:
train_data = WikiText103(
                       split="train",
                       tokenizer=tokenizer,
                       context_length=CONTEXT_LENGTH,
                       )
test_data = WikiText103(
                      split="test",
                      tokenizer=tokenizer,
                      context_length=CONTEXT_LENGTH,
                      )

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset=train_data,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          )
test_loader = DataLoader(dataset=test_data,
                          batch_size=BATCH_SIZE,
                          shuffle=False,
                          )

In [ ]:
import torch, math
def xavier_uniform(shape: tuple, in_features: int, out_features: int):
    limit = math.sqrt(6.0/(in_features + out_features))
    return -2*limit*torch.rand(shape, requires_grad=True)+limit

In [ ]:
from torch import nn
from torch.nn.parameter import Parameter
import math

class ScribeLM(nn.Module):
    def __init__(self,
                             context_length: int,
                             vocab_size: int,
                             n: int=6,
                             d_model: int=512,
                             d_ff: int=2048,
                             h: int=8,
                             ):
        super().__init__()

        self.context_length = context_length
        self.vocab_size = vocab_size
        self.n = n
        self.d_model = d_model
        self.d_ff = d_ff
        self.h = h
        assert self.d_model%h==0
        self.d_k = self.d_v = d_model//h

        self.embedding =  nn.Embedding(self.vocab_size, self.d_model, sparse=True)#Parameter(xavier_uniform((self.vocab_size, self.d_model), self.vocab_size, self.d_model))

        self.positional_encodings = torch.tile(torch.arange(self.d_model, dtype=torch.float32, device=device), (self.context_length, 1))
        for pos in range(self.context_length):
            self.positional_encodings[pos, ::2] = torch.sin((pos/10_000)**(2*self.positional_encodings[pos, ::2]/self.d_model))
            self.positional_encodings[pos, 1::2] = torch.cos((pos/10_000)**(2*self.positional_encodings[pos, 1::2]/self.d_model))

        self.w_q = Parameter(xavier_uniform((self.n, self.h, self.d_model, self.d_k), self.d_model, self.d_k))
        self.w_k =  Parameter(xavier_uniform((self.n, self.h, self.d_model, self.d_k), self.d_model, self.d_k))
        self.w_v =  Parameter(xavier_uniform((self.n, self.h, self.d_model, self.d_v), self.d_model, self.d_v))
        self.w_o =  Parameter(xavier_uniform((self.n, self.h*self.d_v, self.d_model), self.h*self.d_v, self.d_model))
        self.ff_1 =  Parameter(xavier_uniform((self.n, self.d_model, self.d_ff), self.d_model, self.d_ff))
        self.ff_2 =  Parameter(xavier_uniform((self.n, self.d_ff, self.d_model), self.d_ff, self.d_model))
        self.b_1 =  Parameter(torch.zeros((self.n, self.d_ff), requires_grad=True))
        self.b_2 =  Parameter(torch.zeros((self.n, self.d_model), requires_grad=True))
        self.layernorm_1 = list([nn.LayerNorm(self.d_model, device=device) for _ in range(self.n)])
        self.layernorm_2 = list([nn.LayerNorm(self.d_model, device=device) for _ in range(self.n)])
    def forward(self, x: torch.Tensor):
        tokens = x.detach().clone()

        assert tokens.dim() <= 2

        if tokens.dim() == 1:
            tokens = tokens.unsqueeze(0)
        tokens = tokens[:, :self.context_length]
        batch_size = tokens.shape[0]
        context = tokens.shape[-1]
        assert tokens.shape == (batch_size, context)

        embedded = self.embedding(tokens)
        assert embedded.shape == (batch_size, context, self.d_model)
        embedded = torch.add(embedded, self.positional_encodings[:context])

        for layer in range(self.n):
            tiled_embedded = embedded.unsqueeze(1).repeat(1, self.h, 1, 1)
            assert tiled_embedded.shape == (batch_size, self.h, context, self.d_model)
            q = tiled_embedded @ self.w_q[layer, :, :, :].unsqueeze(0)
            k = tiled_embedded @ self.w_k[layer, :, :, :].unsqueeze(0)
            v = tiled_embedded @ self.w_v[layer, :, :, :].unsqueeze(0)
            assert q.shape == (batch_size, self.h, context, self.d_k)
            mask = torch.triu(torch.ones((1, 1, context, context)), diagonal=1).bool().to(device)
            y = nn.functional.softmax(((q @ k.transpose(-1, -2))/math.sqrt(self.d_k)).masked_fill(mask, -torch.inf), dim=-1) @ v
            assert y.shape == (batch_size, self.h, context, self.d_v)
            y = torch.reshape(y.transpose(-2, -3), (batch_size, context, -1))
            assert y.shape == (batch_size, context, self.d_v*self.h)
            embedded = self.layernorm_1[layer](embedded + y @ self.w_o[layer, :, :].unsqueeze(0))
            assert embedded.shape == (batch_size, context, self.d_model)
            y = torch.maximum(torch.tensor(0), embedded @ self.ff_1[layer, :].unsqueeze(0) + self.b_1[layer, :])
            embedded = self.layernorm_2[layer](embedded + y @ self.ff_2[layer, :].unsqueeze(0) + self.b_2[layer, :])
            assert embedded.shape == (batch_size, context, self.d_model)
        return embedded @ self.embedding.weight.T.unsqueeze(0)

In [ ]:
model = ScribeLM(context_length=CONTEXT_LENGTH,
                 vocab_size=VOCAB_SIZE,
                 n=NUM_LAYERS,
                 d_model=D_MODEL,
                 d_ff=D_FEEDFORWARD,
                 h=NUM_HEADS,
                 )
model.to(device)

In [ ]:
import torch
import math
import wandb
from torch import nn

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

scaler = torch.amp.GradScaler(device=device)

if not torch.cuda.is_bf16_supported():
  print("Warning: bfloat16 not supported")

autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

run = wandb.init(
  entity="jschaller2028-personal",
  project="ScribeLM",
  name="test",
  config={
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "context_length": CONTEXT_LENGTH,
    "vocab_size": VOCAB_SIZE,
    "num_layers": NUM_LAYERS,
    "d_model": D_MODEL,
    "d_feedforward": D_FEEDFORWARD,
    "num_heads": NUM_HEADS,
    "lr": LR,
  }
)

print(f"|{"Mode":^10}|{"Epoch":^10}|{"Batch":^10}|{"Loss":^10}|{"Accuracy":^10}|{"PPL":^20}|")

total_steps = 0

for epoch in range(EPOCHS):
    for i, data in enumerate(train_loader):
        x, y = data
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device, dtype=autocast_dtype):
            preds = model(x)
            loss = loss_fn(preds.permute(0, 2, 1), y)

        optimizer.zero_grad()
        if torch.cuda.is_bf16_supported():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_steps += 1
        if (i+1)%500==0:
            train_loss = loss.item()
            train_accuracy = float(torch.sum(torch.argmax(preds, dim=-1)==y)/y.numel()*100.0)
            train_ppl = float(math.exp(loss.item()))
            print(f"|{"TRAIN":^10}|{epoch+1:^10}|{f"{i+1}/{len(train_loader)}":^10}|{round(train_loss, 5):^10}|{round(train_accuracy, 5):^10}|{round(train_ppl, 5):^20}|")
            torch.save(model.state_dict(), f"/content/drive/MyDrive/wikitextmodel-cl{CONTEXT_LENGTH}-{total_steps}steps.pth")
            with torch.inference_mode():
                test_loss = 0.0
                test_accuracy = 0.0
                test_ppl = 0.0
                for i, data in enumerate(test_loader):
                    x, y = data
                    x, y = x.to(device), y.to(device)
                    with torch.amp.autocast(device, dtype=autocast_dtype):
                        preds = model(x)
                        loss = loss_fn(preds.permute(0, 2, 1), y)
                    test_loss += loss.item()
                    accuracy = float(torch.sum(torch.argmax(preds, dim=-1)==y)/y.numel()*100.0)
                    test_accuracy += accuracy
                    ppl = float(math.exp(loss.item()))
                    test_ppl += ppl
            test_loss /= len(test_loader)
            test_accuracy /= len(test_loader)
            test_ppl /= len(test_loader)
            print(f"|{"TEST":^10}|{epoch+1:^10}|{"":^10}|{round(test_loss, 5):^10}|{round(test_accuracy, 5):^10}|{round(test_ppl, 5):^20}|")
            run.log(
            {
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "train_perplexity": train_ppl,
                "test_loss": test_loss,
                "test_accuracy": test_accuracy,
                "test_perplexity": test_ppl,
            }
            )
run.finish()